# RF 기반 분류 앙상블 비교

튜닝 RF를 비교 기준으로 고정하고 TabICL 없이 단일 모델·Soft Voting·Stacking을 비교한다.

- 기존 13개 입력, 원본 448행, 행별 Unknown 총 4개, 마스킹 10세트, 그룹 80:20 분할 30회를 그대로 사용한다.
- RF는 **각 회차 Train에서 선정했던 best params**를 재사용한다. 1회차 설정을 전체 회차에 적용하지 않는다.
- LR·ExtraTrees·CatBoost는 각 회차 Train 안의 그룹 5-Fold에서 Brier로 설정을 선택한다.
- Stacking은 그룹 3-Fold에서 학습에 쓰지 않은 행의 확률을 만들어 LogisticRegression을 학습한다.
- 임계값은 0.5다. Accuracy·AUC·Brier·Precision·Recall·F1·FP/FN·FPR을 모두 비교한다.
- 미리 정한 7개 후보만 평가한다. Test를 보고 조합·가중치·탐색 범위를 추가하지 않는다.
- 최종 후보는 튜닝 RF 대비 Brier·Accuracy·AUC·FP·FPR이 모두 나빠지지 않고 하나 이상 개선되는 모델 중 Brier가 가장 작은 모델이다. 그런 모델이 없으면 RF를 유지한다.
- 이 선택은 이미 살펴본 데이터의 반복 비교 결과를 사용한다. 최종 미사용 Test 성적이나 실제 서비스 성능으로 단정하지 않는다.
- 저장할 모델은 기존과 같은 1회차 Train 학습본이다. 기존 산출물과 백엔드·AWS 모델은 교체하지 않는다.

학습·추론은 CPU에서 수행한다. 사용 중인 sklearn 트리 모델과 CatBoost는 MPS를 지원하지 않는다.

실행 기록은 동일한 원본 CSV·베이스라인·튜닝 RF를 복사한 임시 작업 폴더에서 새 커널로 전체 재실행했다. 아래 상대 저장 경로와 산출물 해시는 재실행 파일 기준이며, 실행 시간 등 메타데이터가 달라 기존 파일과 해시는 다를 수 있다.
기존 로컬·배포 산출물은 덮어쓰지 않았다. 기존 배포본의 SHA-256은 `609c5d63b201fcb125cca9cddc2fcbe229f76d3ebf0a1417466d027248b17681`이며 아래 이번 실행 산출물 해시와 구분한다.


## 1. 라이브러리


In [1]:
import hashlib
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython import get_ipython
from IPython.display import display
from IPython.utils.capture import capture_output
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, ParameterGrid, StratifiedGroupKFold
from sklearn.pipeline import Pipeline

## 2. 전처리 재사용

기존 전처리만 실행한다. 예전 7:3 분할은 사용하지 않고 전체 원본과 마스킹 10세트를 가져온다.
원본 CSV 경로는 `SALESLUV_B2B_DATA_PATH` 설정을 따른다.


In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    preprocessing_notebook = current_dir / "deal_data_preprocessing.ipynb"
elif current_dir.name == "backend" and (current_dir / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "notebooks" / "deal_data_preprocessing.ipynb"
elif (current_dir / "backend" / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "backend" / "notebooks" / "deal_data_preprocessing.ipynb"
else:
    raise RuntimeError("backend/notebooks 폴더를 찾을 수 없습니다.")

ipython = get_ipython()
assert ipython is not None, "Jupyter 커널에서 실행해야 합니다."
# 전처리에서 출력하는 원본 영업 행은 이 노트북 출력에 다시 저장하지 않는다.
with capture_output():
    ipython.run_line_magic("run", str(preprocessing_notebook))

MODEL_FEATURE_NAMES = ipython.user_ns["MODEL_FEATURE_NAMES"]
CATEGORY_VALUES = ipython.user_ns["CATEGORY_VALUES"]
X_categorical = ipython.user_ns["X_categorical"]
X_all_masked_sets = ipython.user_ns["X_all_masked_sets"]
y = ipython.user_ns["y"]
input_group_ids = ipython.user_ns["input_group_ids"]
SOURCE_SHA256 = ipython.user_ns["SOURCE_SHA256"]
UNKNOWN_COLUMNS_PER_ROW = ipython.user_ns["UNKNOWN_COLUMNS_PER_ROW"]

assert X_categorical.shape == (448, 13)
assert X_categorical.index.is_unique
assert X_categorical.index.equals(y.index)
assert y.index.equals(input_group_ids.index)
assert list(X_categorical.columns) == list(MODEL_FEATURE_NAMES)
assert set(y.unique()) == {0, 1}
assert len(X_all_masked_sets) == 10
assert UNKNOWN_COLUMNS_PER_ROW == 4
for masked_data in X_all_masked_sets.values():
    assert masked_data.index.equals(y.index)
    assert list(masked_data.columns) == list(MODEL_FEATURE_NAMES)
    assert masked_data.eq("Unknown").sum(axis=1).eq(UNKNOWN_COLUMNS_PER_ROW).all()
    assert (X_categorical.eq("Unknown") <= masked_data.eq("Unknown")).all().all()

# 각 마스킹 세트의 원본 행 순서와 정답 순서를 맞춘다.
X_all_raw = pd.concat(X_all_masked_sets, names=["mask_set", "original_row_id"])
y_all_masked = pd.concat(
    {set_name: y for set_name in X_all_masked_sets},
    names=["mask_set", "original_row_id"],
)
all_original_row_ids = X_all_raw.index.get_level_values("original_row_id")
assert X_all_raw.index.equals(y_all_masked.index)
assert X_all_raw.shape == (4480, 13)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 180)
print(f"원본 {len(y)}행, 입력 {len(MODEL_FEATURE_NAMES)}개")
print(f"동일 입력 {input_group_ids.nunique()}그룹, 마스킹 {len(X_all_masked_sets)}세트")

원본 448행, 입력 13개
동일 입력 198그룹, 마스킹 10세트


### 해석

원본 순서와 범주를 유지하고, 중복 입력과 마스킹 변형은 원본 입력 그룹으로 묶는다.
마스킹 4,480행을 독립된 새 영업 사례로 취급하지 않는다.


## 3. 튜닝 RF와 기존 평가 분할 불러오기


In [3]:
artifact_dir = preprocessing_notebook.parents[1] / "pipeline" / "artifacts"
tuned_artifact_path = artifact_dir / "deal-paper-rf-tuned-v1.joblib"
baseline_artifact_path = artifact_dir / "deal-paper-rf-baseline-v1.joblib"
assert tuned_artifact_path.exists(), "먼저 RF 튜닝 노트북을 실행해야 합니다."
assert baseline_artifact_path.exists(), "RF 베이스라인 파일이 필요합니다."
tuned_sha256 = hashlib.sha256(tuned_artifact_path.read_bytes()).hexdigest()
baseline_sha256 = hashlib.sha256(baseline_artifact_path.read_bytes()).hexdigest()
tuned_bundle = joblib.load(tuned_artifact_path)

assert tuned_bundle["model_version"] == "deal-paper-rf-tuned-v1"
assert tuned_bundle["baseline_artifact_sha256"] == baseline_sha256
assert tuned_bundle["source_sha256"] == SOURCE_SHA256
assert tuned_bundle["model_feature_names"] == list(MODEL_FEATURE_NAMES)
assert tuned_bundle["category_values"] == {
    column: list(values) for column, values in CATEGORY_VALUES.items()
}
for package, saved_version in tuned_bundle["versions"].items():
    assert version(package) == saved_version, f"{package} 버전이 기준 실험과 다릅니다."

evaluation_splits = tuned_bundle["evaluation"]["splits"]
rf_selection_results = tuned_bundle["tuning"]["selection_results"]
saved_rf_masks = (
    tuned_bundle["evaluation"]["mask_results"]
    .query("model == 'RandomForest_tuned'")
    .set_index(["repeat", "mask_set"])
)
CLASSIFICATION_THRESHOLD = tuned_bundle["classification_threshold"]
REFERENCE_REPEAT = tuned_bundle["reference_repeat"]
RANDOM_STATE = 1
INNER_CV_FOLDS = 5
STACKING_FOLDS = 3
SCORING = "neg_brier_score"
assert CLASSIFICATION_THRESHOLD == 0.5
assert REFERENCE_REPEAT == 1
assert len(evaluation_splits) == 30
assert set(rf_selection_results.index) == set(range(1, 31))
assert tuned_bundle["evaluation"]["masking_set_count"] == len(X_all_masked_sets) == 10
assert tuned_bundle["evaluation"]["unknown_columns_per_row"] == UNKNOWN_COLUMNS_PER_ROW == 4
display(tuned_bundle["evaluation"]["comparison_mean"].round(6))

,accuracy,auc,brier,precision,recall,f1,fp,fn,tn,tp,fpr
model,,,,,,,,,,,
RandomForest_base,0.695847,0.740949,0.209706,0.695281,0.740104,0.714223,15.593333,12.543333,27.440000,36.256667,0.357257
RandomForest_tuned,0.733833,0.775560,0.190198,0.706008,0.835410,0.763253,16.766667,7.636667,26.266667,41.163333,0.384271


### 해석

기준 RF와 튜닝 RF의 성적을 다시 확인한다. 현재 비교 기준은 `RandomForest_tuned`다.
이미 학습한 RF를 다른 회차 Test에 바로 적용하지 않고 설정만 복사해 해당 회차 Train으로 새로 학습한다.
이 프로젝트에서 만든 로컬 joblib만 읽으며 외부의 임의 모델 파일로 대체하지 않는다.


## 4. 단일 모델

### 4.1 RandomForest

여러 나무의 평균으로 비선형 관계를 학습한다. 직전 튜닝에서 각 회차 Train으로 선정한 설정을 사용한다.


In [4]:
model_rf = clone(tuned_bundle["model"])
encoder_template = clone(model_rf.named_steps["onehot"])


def make_one_hot_model(classifier):
    """기존 RF와 같은 범주 순서·원핫 인코딩을 사용하는 모델."""
    return Pipeline([("onehot", clone(encoder_template)), ("classifier", classifier)])


assert not hasattr(model_rf.named_steps["classifier"], "estimators_")

### 4.2 LogisticRegression

트리와 다른 선형 판단을 더한다. 규제 강도와 클래스 가중치 8개 조합을 Train 내부에서 비교한다.


In [5]:
model_lr = make_one_hot_model(
    LogisticRegression(max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE)
)
param_lr = {
    "classifier__C": [0.01, 0.03, 0.1, 1.0],
    "classifier__class_weight": [None, "balanced"],
}

### 4.3 ExtraTrees

RF보다 분기 기준에 무작위성을 더하는 트리 모델이다.
나무 수는 300개로 고정하고 깊이·최소 잎 크기·분기 피처 수 18개 조합을 비교한다.


In [6]:
model_et = make_one_hot_model(
    ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=1)
)
param_et = {
    "classifier__max_depth": [4, 8, 12],
    "classifier__min_samples_leaf": [3, 6, 12],
    "classifier__max_features": ["sqrt", 0.5],
}

### 4.4 CatBoost

원본 범주형 입력을 직접 사용하는 부스팅 모델이다. Unknown도 하나의 범주로 받는다.
반복 수는 200으로 고정하고 깊이·학습률·규제 18개 조합을 비교한다. 원핫 인코딩은 적용하지 않는다.


In [7]:
model_catboost = CatBoostClassifier(
    iterations=200,
    cat_features=tuple(MODEL_FEATURE_NAMES),
    loss_function="Logloss",
    random_seed=RANDOM_STATE,
    thread_count=1,
    verbose=False,
    allow_writing_files=False,
)
param_catboost = {
    "depth": [3, 4, 6],
    "learning_rate": [0.01, 0.05, 0.1],
    "l2_leaf_reg": [3.0, 7.0],
}
additional_models = {
    "LogisticRegression": (model_lr, param_lr),
    "ExtraTrees": (model_et, param_et),
    "CatBoost": (model_catboost, param_catboost),
}
search_counts = {name: len(ParameterGrid(param)) for name, (_, param) in additional_models.items()}
assert search_counts == {"LogisticRegression": 8, "ExtraTrees": 18, "CatBoost": 18}
print(f"추가 모델 탐색: {search_counts}")
print(f"검증 학습: {sum(search_counts.values()) * INNER_CV_FOLDS * len(evaluation_splits):,}회")

추가 모델 탐색: {'LogisticRegression': 8, 'ExtraTrees': 18, 'CatBoost': 18}
검증 학습: 6,600회


### 해석

추가 모델도 과거의 전역 best params를 그대로 가져오지 않고 현재 회차 Train 안에서 설정을 고른다.
하이퍼파라미터 선택 기준은 Brier지만, LR·CatBoost의 기본 학습 손실이나 트리 분기 기준 자체를 Brier로 바꾸는 것은 아니다.


## 5. 비교할 분류 앙상블

Soft Voting은 모델들의 Won 확률을 같은 비중으로 평균한다.
Stacking은 Train 내부 그룹 3-Fold의 예측 확률로 LR 결합 모델을 학습한다. 임계값은 모두 0.5다.

OOF를 만들 때는 해당 회차 Train에서 고른 파라미터를 고정한다. 파라미터 탐색까지 OOF 안에서 다시 반복하지 않으므로
OOF 자체를 편향 없는 검증 성적으로 보고하지 않는다. 성능 평가는 분리된 바깥 Test에서만 한다.


In [8]:
base_names = ("RandomForest_tuned", "LogisticRegression", "ExtraTrees", "CatBoost")
voting_members = {
    "SoftVoting_RF_LR_CatBoost": (
        "RandomForest_tuned",
        "LogisticRegression",
        "CatBoost",
    ),
    "SoftVoting_RF_LR_ET_CatBoost": base_names,
}
STACKING_NAME = "Stacking_LR"
candidate_names = (*base_names, *voting_members, STACKING_NAME)
assert len(candidate_names) == 7
display(
    pd.DataFrame(
        [{"모델": name, "구성": " + ".join(members)} for name, members in voting_members.items()]
        + [{"모델": STACKING_NAME, "구성": "4개 모델의 확률 → LogisticRegression(C=0.1)"}]
    )
)

,모델,구성
0,SoftVoting_RF_LR_CatBoost,RandomForest_tuned + LogisticRegression + CatB...
1,SoftVoting_RF_LR_ET_CatBoost,RandomForest_tuned + LogisticRegression + Extr...
2,Stacking_LR,4개 모델의 확률 → LogisticRegression(C=0.1)


## 6. 같은 30개 분할에서 튜닝·앙상블 학습·평가


In [9]:
def positive_probability(estimator, X):
    """Won=1 열의 확률을 반환한다."""
    return estimator.predict_proba(X)[:, list(estimator.classes_).index(1)]


def calculate_metrics(y_true, probability):
    """0.5 임계값으로 확률·분류 성능과 FP/FPR을 함께 계산한다."""
    assert np.isfinite(probability).all()
    assert ((probability >= 0) & (probability <= 1)).all()
    prediction = (probability >= CLASSIFICATION_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, prediction),
        "auc": roc_auc_score(y_true, probability),
        "brier": brier_score_loss(y_true, probability),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "tp": int(tp),
        "fpr": float(fp / (fp + tn)),
    }


mask_result_rows = []
repeat_rows = []
selection_rows = []
search_result_frames = []
ensemble_started = perf_counter()

for repeat, (train_positions, test_positions) in enumerate(evaluation_splits, start=1):
    repeat_started = perf_counter()
    train_row_ids = y.index[train_positions]
    test_row_ids = y.index[test_positions]
    assert set(train_row_ids).isdisjoint(test_row_ids)
    assert set(train_row_ids) | set(test_row_ids) == set(y.index)
    assert set(input_group_ids.iloc[train_positions]).isdisjoint(
        input_group_ids.iloc[test_positions]
    )
    assert set(y.iloc[train_positions].unique()) == {0, 1}
    assert set(y.iloc[test_positions].unique()) == {0, 1}
    train_mask = all_original_row_ids.isin(train_row_ids)
    X_repeat_train = X_all_raw.loc[train_mask]
    y_repeat_train = y_all_masked.loc[train_mask]
    repeat_original_ids = X_repeat_train.index.get_level_values("original_row_id")
    repeat_groups = input_group_ids.loc[repeat_original_ids].to_numpy()
    assert X_repeat_train.index.equals(y_repeat_train.index)
    assert len(X_repeat_train) == len(train_positions) * 10
    assert pd.Series(repeat_original_ids).value_counts().eq(10).all()

    search_splits = list(
        StratifiedGroupKFold(
            n_splits=INNER_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE
        ).split(X_repeat_train, y_repeat_train, groups=repeat_groups)
    )
    stacking_splits = list(
        StratifiedGroupKFold(
            n_splits=STACKING_FOLDS, shuffle=True, random_state=RANDOM_STATE
        ).split(X_repeat_train, y_repeat_train, groups=repeat_groups)
    )
    for splits in (search_splits, stacking_splits):
        for inner_train, inner_valid in splits:
            assert set(repeat_groups[inner_train]).isdisjoint(repeat_groups[inner_valid])
            assert set(repeat_original_ids[inner_train]).isdisjoint(
                repeat_original_ids[inner_valid]
            )
            assert set(y_repeat_train.iloc[inner_train].unique()) == {0, 1}
            assert set(y_repeat_train.iloc[inner_valid].unique()) == {0, 1}
        coverage = np.bincount(
            np.concatenate([valid for _, valid in splits]), minlength=len(X_repeat_train)
        )
        assert (coverage == 1).all()

    # RF는 해당 회차의 Train에서 이미 선정한 설정을 재사용한다.
    rf_params = rf_selection_results.loc[repeat, "best_params"]
    repeat_models = {"RandomForest_tuned": clone(model_rf).set_params(**rf_params)}
    selection_rows.append(
        {
            "repeat": repeat,
            "model": "RandomForest_tuned",
            "cv_brier": rf_selection_results.loc[repeat, "tuned_cv_brier"],
            "best_params": rf_params,
        }
    )
    for name, (template, param) in additional_models.items():
        search = GridSearchCV(
            template,
            param,
            cv=search_splits,
            scoring=SCORING,
            n_jobs=-1,
            refit=False,
            return_train_score=True,
            error_score="raise",
        )
        search.fit(X_repeat_train, y_repeat_train)
        assert len(search.cv_results_["params"]) == search_counts[name]
        repeat_models[name] = clone(template).set_params(**search.best_params_)
        selection_rows.append(
            {
                "repeat": repeat,
                "model": name,
                "cv_brier": -search.best_score_,
                "best_params": search.best_params_,
            }
        )
        search_result_frames.append(
            pd.DataFrame(
                {
                    "repeat": repeat,
                    "model": name,
                    "rank": search.cv_results_["rank_test_score"],
                    "params": search.cv_results_["params"],
                    "cv_brier": -search.cv_results_["mean_test_score"],
                    "cv_brier_std": search.cv_results_["std_test_score"],
                    "train_brier": -search.cv_results_["mean_train_score"],
                }
            )
        )
    assert tuple(repeat_models) == base_names

    # sklearn이 전체 Train의 단일 모델과 그룹 OOF 기반 메타 모델을 함께 학습한다.
    fitted_stack = StackingClassifier(
        estimators=list(repeat_models.items()),
        final_estimator=LogisticRegression(
            C=0.1, max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE
        ),
        cv=stacking_splits,
        stack_method="predict_proba",
        passthrough=False,
        n_jobs=-1,
    ).fit(X_repeat_train, y_repeat_train)
    fitted_models = dict(zip(base_names, fitted_stack.estimators_, strict=True))
    assert all(list(estimator.classes_) == [0, 1] for estimator in fitted_models.values())
    assert fitted_stack.stack_method_ == ["predict_proba"] * len(base_names)
    repeat_mask_metrics = {name: [] for name in candidate_names}

    for set_name, masked_data in X_all_masked_sets.items():
        X_repeat_test = masked_data.iloc[test_positions]
        y_repeat_test = y.iloc[test_positions]
        assert X_repeat_test.index.equals(y_repeat_test.index)
        probability_matrix = pd.DataFrame(
            {
                name: positive_probability(estimator, X_repeat_test)
                for name, estimator in fitted_models.items()
            }
        )
        probabilities = {name: probability_matrix[name].to_numpy() for name in base_names}
        for name, members in voting_members.items():
            probabilities[name] = probability_matrix[list(members)].mean(axis=1).to_numpy()
        probabilities[STACKING_NAME] = positive_probability(
            fitted_stack.final_estimator_, probability_matrix.to_numpy()
        )
        for name, probability in probabilities.items():
            metrics = calculate_metrics(y_repeat_test, probability)
            if name == "RandomForest_tuned":
                np.testing.assert_allclose(
                    list(metrics.values()),
                    saved_rf_masks.loc[(repeat, set_name), list(metrics)].to_numpy(),
                    rtol=1e-10,
                    atol=1e-12,
                )
            repeat_mask_metrics[name].append(metrics)
            mask_result_rows.append(
                {"repeat": repeat, "model": name, "mask_set": set_name, **metrics}
            )

    for name, rows in repeat_mask_metrics.items():
        mean_metrics = pd.DataFrame(rows).mean().to_dict()
        repeat_rows.append(
            {
                "repeat": repeat,
                "model": name,
                "train_rows": len(train_positions),
                "test_rows": len(test_positions),
                **mean_metrics,
            }
        )

    if repeat == REFERENCE_REPEAT:
        reference_stack = fitted_stack
        reference_base_models = fitted_models
        reference_stacking_splits = stacking_splits
        reference_train_positions = train_positions.copy()
        reference_test_positions = test_positions.copy()

    print(
        f"{repeat:02d}/30 완료 | 추가 모델 44개 설정 × 5-Fold, "
        f"7개 후보 × 10마스킹 평가 | {perf_counter() - repeat_started:.1f}초",
        flush=True,
    )

ensemble_seconds = perf_counter() - ensemble_started
mask_results = pd.DataFrame(mask_result_rows)
repeat_results = pd.DataFrame(repeat_rows)
selection_results = pd.DataFrame(selection_rows)
search_results = pd.concat(search_result_frames, ignore_index=True)
metric_names = list(mean_metrics)
assert len(mask_results) == 30 * 7 * 10
assert len(repeat_results) == 30 * 7
assert len(selection_results) == 30 * 4
assert len(search_results) == 30 * 44
assert mask_results.groupby(["model", "repeat"]).size().eq(10).all()
assert repeat_results.groupby("model")["repeat"].nunique().eq(30).all()
np.testing.assert_allclose(
    mask_results.groupby(["model", "repeat"])[metric_names].mean(),
    repeat_results.set_index(["model", "repeat"])[metric_names].sort_index(),
)
np.testing.assert_allclose(
    repeat_results[["fp", "fn", "tn", "tp"]].sum(axis=1),
    repeat_results["test_rows"],
)
assert np.isfinite(repeat_results[metric_names].to_numpy()).all()

01/30 완료 | 추가 모델 44개 설정 × 5-Fold, 7개 후보 × 10마스킹 평가 | 12.5초


02/30 완료 | 추가 모델 44개 설정 × 5-Fold, 7개 후보 × 10마스킹 평가 | 11.7초


03/30 완료 | 추가 모델 44개 설정 × 5-Fold, 7개 후보 × 10마스킹 평가 | 13.1초


04/30 완료 | 추가 모델 44개 설정 × 5-Fold, 7개 후보 × 10마스킹 평가 | 11.7초


05/30 완료 | 추가 모델 44개 설정 × 5-Fold, 7개 후보 × 10마스킹 평가 | 14.1초


...

26/30 완료 | 추가 모델 44개 설정 × 5-Fold, 7개 후보 × 10마스킹 평가 | 10.7초


27/30 완료 | 추가 모델 44개 설정 × 5-Fold, 7개 후보 × 10마스킹 평가 | 12.7초


28/30 완료 | 추가 모델 44개 설정 × 5-Fold, 7개 후보 × 10마스킹 평가 | 11.3초


29/30 완료 | 추가 모델 44개 설정 × 5-Fold, 7개 후보 × 10마스킹 평가 | 12.4초


30/30 완료 | 추가 모델 44개 설정 × 5-Fold, 7개 후보 × 10마스킹 평가 | 12.1초


### 해석

각 회차에서 파라미터 선택과 Stacking 학습을 마친 뒤 같은 Test 10세트를 평가했다.
RF의 300개 마스킹 평가를 직전 기록과 대조했으므로 다른 RF 설정이나 분할을 비교하지 않았다.
Train 내부 OOF는 메타 모델 학습용이며 최종 성능으로 보고하지 않는다.


## 7. 전체 모델 비교와 RF 대비 변화


In [10]:
comparison = repeat_results.groupby("model", sort=False)[metric_names].mean()
comparison_std = repeat_results.groupby("model", sort=False)[metric_names].std()
comparison_all = pd.concat(
    [
        tuned_bundle["evaluation"]["comparison_mean"].loc[["RandomForest_base"], metric_names],
        comparison,
    ]
).sort_values("brier", kind="stable")
rf_mean = comparison.loc["RandomForest_tuned"]
delta = comparison.subtract(rf_mean)
display(comparison_all.round(6))
display(delta[["accuracy", "auc", "brier", "fp", "fn", "fpr"]].round(6))
display(comparison_std.round(6))

rf_by_repeat = repeat_results.query("model == 'RandomForest_tuned'").set_index("repeat")[
    metric_names
]
improvement_rows = []
for name in candidate_names:
    if name == "RandomForest_tuned":
        continue
    candidate_by_repeat = repeat_results.loc[repeat_results["model"].eq(name)].set_index("repeat")[
        metric_names
    ]
    paired_delta = candidate_by_repeat - rf_by_repeat
    improvement_rows.append(
        {
            "model": name,
            "Brier_개선": int(paired_delta["brier"].lt(-1e-12).sum()),
            "Accuracy_개선": int(paired_delta["accuracy"].gt(1e-12).sum()),
            "AUC_개선": int(paired_delta["auc"].gt(1e-12).sum()),
            "FP_감소": int(paired_delta["fp"].lt(-1e-12).sum()),
            "FPR_감소": int(paired_delta["fpr"].lt(-1e-12).sum()),
        }
    )
improvement_counts = pd.DataFrame(improvement_rows).set_index("model")
display(improvement_counts)
print(f"전체 추가 튜닝·앙상블·평가 시간: {ensemble_seconds:.1f}초")

,accuracy,auc,brier,precision,recall,f1,fp,fn,tn,tp,fpr
model,,,,,,,,,,,
Stacking_LR,0.735980,0.778318,0.187965,0.711869,0.825865,0.762654,16.150000,8.090000,26.883333,40.710000,0.370101
SoftVoting_RF_LR_ET_CatBoost,0.736531,0.778727,0.188461,0.709887,0.832292,0.764364,16.420000,7.756667,26.613333,41.043333,0.376246
SoftVoting_RF_LR_CatBoost,0.735595,0.778322,0.188502,0.709257,0.830762,0.763363,16.413333,7.846667,26.620000,40.953333,0.376428
ExtraTrees,0.732910,0.776948,0.189994,0.706043,0.831085,0.761563,16.680000,7.823333,26.353333,40.976667,0.382328
RandomForest_tuned,0.733833,0.775560,0.190198,0.706008,0.835410,0.763253,16.766667,7.636667,26.266667,41.163333,0.384271
LogisticRegression,0.727850,0.772088,0.190529,0.707623,0.811970,0.754154,16.086667,8.956667,26.946667,39.843333,0.369847
CatBoost,0.738872,0.773243,0.190599,0.710247,0.838148,0.766934,16.536667,7.403333,26.496667,41.396667,0.379343
RandomForest_base,0.695847,0.740949,0.209706,0.695281,0.740104,0.714223,15.593333,12.543333,27.440000,36.256667,0.357257


,accuracy,auc,brier,fp,fn,fpr
model,,,,,,
RandomForest_tuned,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LogisticRegression,-0.005982,-0.003472,0.000331,-0.680000,1.320000,-0.014424
ExtraTrees,-0.000923,0.001388,-0.000204,-0.086667,0.186667,-0.001943
CatBoost,0.005040,-0.002316,0.000401,-0.230000,-0.233333,-0.004928
SoftVoting_RF_LR_CatBoost,0.001763,0.002762,-0.001696,-0.353333,0.210000,-0.007843
SoftVoting_RF_LR_ET_CatBoost,0.002698,0.003167,-0.001737,-0.346667,0.120000,-0.008025
Stacking_LR,0.002147,0.002758,-0.002233,-0.616667,0.453333,-0.014170


,accuracy,auc,brier,precision,recall,f1,fp,fn,tn,tp,fpr
model,,,,,,,,,,,
RandomForest_tuned,0.044744,0.047202,0.017526,0.065069,0.063904,0.053539,5.366328,2.486582,3.926157,13.712403,0.092293
LogisticRegression,0.044497,0.048541,0.017499,0.062441,0.063428,0.051564,4.923535,3.560674,4.047710,12.592271,0.083741
ExtraTrees,0.047820,0.048712,0.017976,0.065635,0.067888,0.056108,5.395298,2.729618,4.027041,13.766769,0.093768
CatBoost,0.045392,0.048249,0.019962,0.061634,0.072093,0.054689,5.431103,2.899166,4.208570,14.039488,0.095657
SoftVoting_RF_LR_CatBoost,0.043867,0.048719,0.018080,0.063046,0.064940,0.053151,5.258342,2.753773,3.990713,13.582893,0.090613
SoftVoting_RF_LR_ET_CatBoost,0.045438,0.048656,0.018024,0.064406,0.065323,0.054087,5.357199,2.681184,3.984862,13.684869,0.093038
Stacking_LR,0.045607,0.048773,0.018804,0.063889,0.066957,0.053665,5.348912,2.954523,4.056420,13.596484,0.093211


,Brier_개선,Accuracy_개선,AUC_개선,FP_감소,FPR_감소
model,,,,,
LogisticRegression,13,10,13,19,19
ExtraTrees,14,9,17,14,14
CatBoost,16,18,12,13,13
SoftVoting_RF_LR_CatBoost,22,18,20,17,17
SoftVoting_RF_LR_ET_CatBoost,24,15,22,15,15
Stacking_LR,23,15,23,20,20


전체 추가 튜닝·앙상블·평가 시간: 376.2초


### 해석

`RandomForest_base`는 튜닝 전 기준 기록이며, 나머지 7개가 이번 실험이다.
표는 10개 마스킹 평균을 다시 30회 평균한 값이다. 반복들이 겹치므로 독립적인 300회 실험이나 통계적 유의성으로 해석하지 않는다.

차이는 후보에서 튜닝 RF를 뺀 값이다. Accuracy·AUC는 양수, Brier·FP·FPR은 음수면 개선이다.
FP/FN은 평균 건수다. 실제 Lost 중 Won으로 잘못 표시한 비율인 FPR도 함께 본다.

이번 결과에서 Stacking_LR은 튜닝 RF 대비 Accuracy 73.38% → 73.60%, AUC 0.7756 → 0.7783, Brier 0.1902 → 0.1880으로 소폭 개선됐다.
평균 FP는 16.77 → 16.15건, FPR은 38.43% → 37.01%로 감소했지만 FN은 7.64 → 8.09건으로 증가하고 Recall은 83.54% → 82.59%로 낮아졌다.
CatBoost는 Accuracy, 4모델 Voting은 AUC, Stacking은 Brier가 가장 좋았다. Stacking이 모든 지표에서 1위인 것은 아니다.
튜닝 전 RF의 FP 15.59건과 비교하면 Stacking의 FP 16.15건은 여전히 많다. 평균 개선 폭이 작으므로 확정적 우월이나 배포 성능으로 단정하지 않는다.


## 8. 지표 간 손해를 강요하지 않는 후보 선정


In [11]:
# 계산 오차 범위의 차이는 동률로 보고, 한 지표를 희생하는 자동 교체는 하지 않는다.
tolerance = 1e-12
no_worse = (
    comparison["brier"].le(rf_mean["brier"] + tolerance)
    & comparison["accuracy"].ge(rf_mean["accuracy"] - tolerance)
    & comparison["auc"].ge(rf_mean["auc"] - tolerance)
    & comparison["fp"].le(rf_mean["fp"] + tolerance)
    & comparison["fpr"].le(rf_mean["fpr"] + tolerance)
)
strictly_better = (
    comparison["brier"].lt(rf_mean["brier"] - tolerance)
    | comparison["accuracy"].gt(rf_mean["accuracy"] + tolerance)
    | comparison["auc"].gt(rf_mean["auc"] + tolerance)
    | comparison["fp"].lt(rf_mean["fp"] - tolerance)
    | comparison["fpr"].lt(rf_mean["fpr"] - tolerance)
)
eligible = comparison.loc[no_worse & strictly_better].sort_values("brier", kind="stable")
selected_name = eligible.index[0] if not eligible.empty else "RandomForest_tuned"
best_brier_name = comparison["brier"].idxmin()
selection_review = comparison[["accuracy", "auc", "brier", "fp", "fpr"]].copy()
selection_review["RF대비_동시개선"] = no_worse & strictly_better
display(selection_review.round(6))
print(f"Brier만 보았을 때 1위: {best_brier_name}")
print(f"여러 지표를 함께 확인한 선정 후보: {selected_name}")
display(selection_results.loc[selection_results["repeat"].eq(REFERENCE_REPEAT)].round(6))

,accuracy,auc,brier,fp,fpr,RF대비_동시개선
model,,,,,,
RandomForest_tuned,0.733833,0.775560,0.190198,16.766667,0.384271,False
LogisticRegression,0.727850,0.772088,0.190529,16.086667,0.369847,False
ExtraTrees,0.732910,0.776948,0.189994,16.680000,0.382328,False
CatBoost,0.738872,0.773243,0.190599,16.536667,0.379343,False
SoftVoting_RF_LR_CatBoost,0.735595,0.778322,0.188502,16.413333,0.376428,True
SoftVoting_RF_LR_ET_CatBoost,0.736531,0.778727,0.188461,16.420000,0.376246,True
Stacking_LR,0.735980,0.778318,0.187965,16.150000,0.370101,True


Brier만 보았을 때 1위: Stacking_LR
여러 지표를 함께 확인한 선정 후보: Stacking_LR


,repeat,model,cv_brier,best_params
0,1,RandomForest_tuned,0.184536,"{'classifier__max_depth': 12, 'classifier__max..."
1,1,LogisticRegression,0.187761,"{'classifier__C': 0.1, 'classifier__class_weig..."
2,1,ExtraTrees,0.183468,"{'classifier__max_depth': 12, 'classifier__max..."
3,1,CatBoost,0.180793,"{'depth': 4, 'l2_leaf_reg': 3.0, 'learning_rat..."


### 해석

Brier가 조금 작아졌다는 이유만으로 Accuracy·AUC·FP/FPR 악화를 자동 수용하지 않는다.
동시 개선 후보가 없으면 RF를 유지하고 모든 비교 수치를 남긴다. 임계값이나 FP 비용비를 새로 정하지 않았다.

후보 종류를 이 30회 평가로 선택했으므로 선정 이후의 완전히 독립된 최종 검증은 남아 있지 않다.
단일 회차 최고 점수의 모델을 고르지 않으며, 외부 실제 미팅 데이터 검증 전에는 배포 성능을 확정하지 않는다.


## 9. 선정 후보 저장과 예측 재검증


In [12]:
known_row = {
    column: next(value for value in CATEGORY_VALUES[column] if value != "Unknown")
    for column in MODEL_FEATURE_NAMES
}
masked_row = {**known_row, **dict.fromkeys(MODEL_FEATURE_NAMES[:4], "Unknown")}
self_check_X = pd.DataFrame(
    [known_row, masked_row, dict.fromkeys(MODEL_FEATURE_NAMES, "Unknown")],
    columns=list(MODEL_FEATURE_NAMES),
)
reference_matrix = np.column_stack(
    [positive_probability(reference_base_models[name], self_check_X) for name in base_names]
)
np.testing.assert_allclose(
    positive_probability(reference_stack, self_check_X),
    positive_probability(reference_stack.final_estimator_, reference_matrix),
    rtol=1e-12,
    atol=1e-12,
)

if selected_name in voting_members:
    # 비교 중에는 확률 평균을 재사용하고, 저장 후보만 sklearn VotingClassifier로 만든다.
    members = voting_members[selected_name]
    selected_model = VotingClassifier(
        estimators=[(name, clone(reference_base_models[name])) for name in members],
        voting="soft",
        n_jobs=-1,
    )
    reference_train_mask = all_original_row_ids.isin(y.index[reference_train_positions])
    selected_model.fit(X_all_raw.loc[reference_train_mask], y_all_masked.loc[reference_train_mask])
    np.testing.assert_allclose(
        positive_probability(selected_model, self_check_X),
        np.mean(
            [positive_probability(reference_base_models[name], self_check_X) for name in members],
            axis=0,
        ),
        rtol=1e-12,
        atol=1e-12,
    )
elif selected_name == STACKING_NAME:
    selected_model = reference_stack
else:
    selected_model = reference_base_models[selected_name]

# 저장할 실제 객체가 비교에 사용한 1회차 Test 확률·지표와 일치하는지 10세트 모두 확인한다.
selected_reference_masks = mask_results.loc[
    mask_results["repeat"].eq(REFERENCE_REPEAT) & mask_results["model"].eq(selected_name)
].set_index("mask_set")
for set_name, X in X_all_masked_sets.items():
    probability = positive_probability(selected_model, X.iloc[reference_test_positions])
    actual_metrics = calculate_metrics(y.iloc[reference_test_positions], probability)
    np.testing.assert_allclose(
        list(actual_metrics.values()),
        selected_reference_masks.loc[set_name, list(actual_metrics)].to_numpy(),
        rtol=1e-10,
        atol=1e-12,
    )

artifact_path = artifact_dir / "deal-paper-rf-ensemble-v1.joblib"
bundle = {
    "schema_version": 1,
    "model_version": "deal-paper-rf-ensemble-v1",
    "model": selected_model,
    "selected_candidate": selected_name,
    "best_brier_candidate": best_brier_name,
    "selection_rule": "no_worse_brier_accuracy_auc_fp_fpr_than_tuned_rf_then_min_brier",
    "selection_scope": "exploratory_selection_after_repeated_test_comparison",
    "model_feature_names": list(MODEL_FEATURE_NAMES),
    "category_values": {column: list(values) for column, values in CATEGORY_VALUES.items()},
    "target": {"Lost": 0, "Won": 1},
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "source_sha256": SOURCE_SHA256,
    "baseline_artifact_sha256": baseline_sha256,
    "tuned_rf_artifact_sha256": tuned_sha256,
    "training_scope": "reference_split_train_only",
    "reference_repeat": REFERENCE_REPEAT,
    "reference_train_positions": reference_train_positions,
    "reference_test_positions": reference_test_positions,
    "training_original_rows": len(reference_train_positions),
    "training_masked_rows": len(reference_train_positions) * len(X_all_masked_sets),
    "reference_test_metrics": selected_reference_masks[metric_names].mean().to_dict(),
    "reference_stacking_splits": reference_stacking_splits,
    "reference_base_params": {
        name: (
            estimator.named_steps["classifier"].get_params()
            if isinstance(estimator, Pipeline)
            else estimator.get_params()
        )
        for name, estimator in reference_base_models.items()
    },
    "tuning": {
        "scoring": SCORING,
        "param_grids": {name: param for name, (_, param) in additional_models.items()},
        "inner_cv_n_splits": INNER_CV_FOLDS,
        "inner_cv_random_state": RANDOM_STATE,
        "selection_results": selection_results,
        "search_results": search_results,
    },
    "evaluation": {
        "splitter": tuned_bundle["evaluation"]["splitter"],
        "split_random_state": tuned_bundle["evaluation"]["split_random_state"],
        "test_group_fraction": tuned_bundle["evaluation"]["test_group_fraction"],
        "repeat_count": len(evaluation_splits),
        "masking_set_count": len(X_all_masked_sets),
        "unknown_columns_per_row": UNKNOWN_COLUMNS_PER_ROW,
        "splits": evaluation_splits,
        "repeat_results": repeat_results,
        "mask_results": mask_results,
        "comparison_mean": comparison,
        "comparison_std": comparison_std,
        "delta_vs_tuned_rf": delta,
        "improvement_counts": improvement_counts,
        "selection_review": selection_review,
        "seconds": ensemble_seconds,
    },
    "versions": {
        name: version(name) for name in ("scikit-learn", "numpy", "pandas", "joblib", "catboost")
    },
}
joblib.dump(bundle, artifact_path)
restored = joblib.load(artifact_path)
np.testing.assert_allclose(
    selected_model.predict_proba(self_check_X),
    restored["model"].predict_proba(self_check_X),
    rtol=1e-12,
    atol=1e-12,
)
assert set(restored["model"].classes_) == {0, 1}
assert len(restored["model_feature_names"]) == 13
assert np.isfinite(restored["model"].predict_proba(self_check_X)).all()
assert hashlib.sha256(tuned_artifact_path.read_bytes()).hexdigest() == tuned_sha256
assert hashlib.sha256(baseline_artifact_path.read_bytes()).hexdigest() == baseline_sha256
print(f"선정 후보: {selected_name}")
print(f"후보 저장: backend/pipeline/artifacts/{artifact_path.name}")
print(f"파일 크기: {artifact_path.stat().st_size / 1024**2:.3f} MiB")
print(f"이번 실행 산출물 SHA-256: {hashlib.sha256(artifact_path.read_bytes()).hexdigest()}")
print(f"합성 3행 Won 확률: {positive_probability(selected_model, self_check_X).tolist()}")
print("RF 재현·내부/외부 그룹 분리·7개 후보 비교·저장 객체의 Test 재현·재로드 검증: 통과")

선정 후보: Stacking_LR
후보 저장: backend/pipeline/artifacts/deal-paper-rf-ensemble-v1.joblib
파일 크기: 16.345 MiB
이번 실행 산출물 SHA-256: b0d87b45c0693cd59b5101be67e18dd5a60d9d7a8fc81fa8e749255564636e39
합성 3행 Won 확률: [0.5625386746948857, 0.3512072797418429, 0.567269464941283]
RF 재현·내부/외부 그룹 분리·7개 후보 비교·저장 객체의 Test 재현·재로드 검증: 통과


### 해석

파일 하나에 선정 후보·입력 계약·설정·전체 비교 결과를 저장했다.
30회 평균은 후보 학습 절차의 비교 성적이고 저장 파일 하나의 성적이 아니다. 저장본의 성적은 `reference_test_metrics`에 따로 있다.

1회차 Train만 학습한 검토용 후보이며 전체 데이터 재학습이나 백엔드 로더 변경은 하지 않았다.
기존 기준 RF·튜닝 RF·배포 모델은 보존했다. 원본 영업 행과 모델 산출물은 Git에 올리지 않는다.

저장된 Stacking의 1회차 Test 성적은 Accuracy 0.716484, AUC 0.732313, Brier 0.206093, FP 15.5건이다.
이 회차에서는 RF의 Accuracy 0.720879, FP 15.4건보다 소폭 나쁘다. 후보 학습 절차의 30회 평균 개선이 저장 파일 하나의 우월성을 보장하지는 않는다.
